# Phase 0 pilot - five-case VLM smoke test (corrected CT preprocessing)

**SMOKE TEST ONLY. This notebook does NOT validate tumor-measurement accuracy.** It checks that the pipeline now
runs on valid (non-blank) images and records exactly how the model behaves, with full raw logging.

- Model: `google/medgemma-1.5-4b-it` (gated; needs accepted terms + HF token in Colab secret `HF_TOKEN`).
- Inputs: `pilot5_images.zip` built locally by `build_pilot_bundle.py` (CT slices, corrected offset 1024).
  **The intensity offset 1024 is an UNVERIFIED hypothesis.**
- Scored cases: 0, 6, 8. Exploratory cases: 2, 4 (QC-flagged). Prompts are identical to Milestone 6.
- **Limitation:** Experiment B states the native pixel spacing, but the model sees an ~896x896 resized image, so the
  stated spacing does not describe the pixels the model sees.
- Fail-closed: nothing runs unless the bundle's SHA-256 hashes match values pinned inside this notebook.

**Run cells top to bottom. Restart the runtime after the install cell, then continue from Step 3.**

## Step 1 - Enable GPU
Runtime -> Change runtime type -> T4 GPU.

In [ ]:
!nvidia-smi

## Step 2 - Install dependencies (versions are recorded in the run metadata)

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes huggingface_hub "pillow==11.3.0"
import transformers, accelerate, bitsandbytes, PIL
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("pillow:", PIL.__version__)

### IMPORTANT - restart the runtime now (Runtime -> Restart session), then continue from Step 3. Do not re-run the install cell.

## Step 3 - Hugging Face login (Colab secret; no token is stored in this notebook)

In [ ]:
from huggingface_hub import login

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    print("Loaded token from Colab secret 'HF_TOKEN'.")
except Exception:
    from getpass import getpass
    hf_token = getpass("Enter your Hugging Face token: ")

login(token=hf_token)
hf_token = None

## Step 4 - Upload and VERIFY the bundle (fail-closed)
Upload exactly `pilot5_images.zip`. The notebook aborts if the bundle is missing, modified, contains extra files, contains any known stale/blank image, or fails any image check. The hashes below are pinned at build time.

In [ ]:
import hashlib, sys, zipfile
from pathlib import Path
from google.colab import files

EXPECTED_MANIFEST_SHA256 = "b4da98950307257b54f3c5c001126753aee1fd8c7890ea8f360297f9d263188f"
EXPECTED_PILOT_LIB_SHA256 = "e63202105eb5f10cab4aba5899413dc059f27fd68052029efcb9580905851f1b"
EXPECTED_BUNDLE_ID = "3cc87e8a4cc6b584"
BUNDLE_ZIP_NAME = "pilot5_images.zip"
BUNDLE_DIR = Path("pilot5_bundle")

BUNDLE_VERIFIED = False
if BUNDLE_DIR.exists():
    raise RuntimeError("pilot5_bundle/ already exists from an earlier run. Use Runtime -> Disconnect and delete runtime, "
                       "then start again, so no stale files can be mixed in.")

uploaded = files.upload()
if list(uploaded) != [BUNDLE_ZIP_NAME]:
    raise RuntimeError(f"Upload exactly one file named {BUNDLE_ZIP_NAME}; got {list(uploaded)}")

with zipfile.ZipFile(BUNDLE_ZIP_NAME) as zf:
    for name in zf.namelist():
        if name.startswith("/") or ".." in Path(name).parts:
            raise RuntimeError(f"unsafe path in zip: {name}")
    zf.extractall(BUNDLE_DIR)

def _sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

if _sha(BUNDLE_DIR / "manifest.json") != EXPECTED_MANIFEST_SHA256:
    raise RuntimeError("manifest.json does not match the hash pinned in this notebook. WRONG OR STALE BUNDLE. Aborting.")
if _sha(BUNDLE_DIR / "pilot_lib.py") != EXPECTED_PILOT_LIB_SHA256:
    raise RuntimeError("pilot_lib.py does not match the hash pinned in this notebook. Aborting.")

sys.path.insert(0, str(BUNDLE_DIR))
import pilot_lib

bundle = pilot_lib.verify_bundle(BUNDLE_DIR, EXPECTED_MANIFEST_SHA256, EXPECTED_PILOT_LIB_SHA256)
if bundle["manifest"]["bundle_id"] != EXPECTED_BUNDLE_ID:
    raise RuntimeError("bundle_id mismatch. Aborting.")
BUNDLE_VERIFIED = True
print("BUNDLE VERIFIED:", EXPECTED_BUNDLE_ID)
print(pilot_lib.BANNER)
for c in bundle["cases"]:
    print(f"  case {c['case_id']} [{c['group']}, qc={c['qc_flag']}] {c['filename']} {c['size_px']} sha256={c['sha256'][:12]}...")

In [ ]:
# Look at exactly what will be sent (no overlays, no outlines). Confirm they are real CT slices, not blank.
from PIL import Image
from IPython.display import display

for c in bundle["cases"]:
    im = Image.open(BUNDLE_DIR / "images" / c["filename"])
    print(f"case {c['case_id']} ({c['group']}) {im.size} {im.mode}")
    display(im.resize((im.size[0] * 2, im.size[1] * 2), Image.NEAREST))

## Step 5 - Load the model ONCE (bf16 first; 4-bit fallback cell below)
The loading mode is recorded in the run metadata. Results from the two modes may differ.

In [ ]:
import gc
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText

MODEL_ID = "google/medgemma-1.5-4b-it"
assert MODEL_ID == pilot_lib.MODEL_ID

for _name in ("model", "processor"):
    if _name in globals():
        del globals()[_name]
gc.collect()
torch.cuda.empty_cache()

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16, device_map="auto")
QUANTIZATION = "bf16"
print("Loaded in bf16.")

### 4-bit fallback (only if the bf16 cell above hit a CUDA out-of-memory error)

In [ ]:
import gc
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

MODEL_ID = "google/medgemma-1.5-4b-it"
assert MODEL_ID == pilot_lib.MODEL_ID

for _name in ("model", "processor"):
    if _name in globals():
        del globals()[_name]
gc.collect()
torch.cuda.empty_cache()

quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.bfloat16)
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, quantization_config=quant_config, device_map="auto")
QUANTIZATION = "nf4-4bit"
print("Loaded in 4-bit (nf4).")

## Step 6 - Record run metadata BEFORE any inference (aborts if incomplete)
Creates a NEW run folder with a unique run id, validates the metadata and persists it. Step 7 refuses to start unless the persisted file exists, is valid and equals what is in memory.

In [ ]:
import datetime, json, platform
import importlib.metadata as importlib_metadata
import torch, transformers, PIL

run_meta = None  # a failure anywhere below must leave NO usable metadata behind for Step 7
if not BUNDLE_VERIFIED:
    raise RuntimeError("bundle not verified")
if "QUANTIZATION" not in globals():
    raise RuntimeError("load the model first (Step 5)")
if not torch.cuda.is_available():
    raise RuntimeError("no GPU available; refusing to run on CPU")

def _ver(name):
    try:
        return importlib_metadata.version(name)
    except Exception:
        return "n/a"

RUN_ID = pilot_lib.new_run_id()
STARTED_UTC = pilot_lib.utc_now()
OUT_DIR = Path(f"pilot5_run_{RUN_ID}")
OUT_DIR.mkdir(parents=True, exist_ok=False)

processor_cfg = json.loads(json.dumps(processor.image_processor.to_dict(), default=str))
env = {
    "model_id": MODEL_ID,
    "model_revision": getattr(model.config, "_commit_hash", None) or "unresolved",
    "quantization": QUANTIZATION,
    "device": f"cuda:{torch.cuda.get_device_name(0)}",
    "python_version": platform.python_version(),
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "accelerate_version": _ver("accelerate"),
    "bitsandbytes_version": _ver("bitsandbytes"),
    "pillow_version": PIL.__version__,
    "processor_image_config": processor_cfg,
    "chat_template_format": "processor.apply_chat_template([{role:user, content:[image, text]}], add_generation_prompt=True, tokenize=True, return_dict=True)",
}
_candidate = pilot_lib.new_run_metadata(bundle["manifest"], bundle["manifest_sha256"], env, RUN_ID, STARTED_UTC)
pilot_lib.validate_run_metadata(_candidate)
pilot_lib.atomic_write_json(OUT_DIR / "run_metadata.json", _candidate)
run_meta = _candidate  # assigned only after validation AND persistence succeeded
print("run_id:", RUN_ID, "| model revision:", env["model_revision"], "| quantization:", QUANTIZATION)
print("processor image config:", json.dumps(processor_cfg, indent=2)[:1500])

## Step 7 - Inference (10 planned calls: 5 cases x experiments A and B)
Gate first: refuses to run unless the persisted metadata is valid and equals the in-memory metadata, and the run folder holds nothing else (no earlier raw log, results or lock). A rerun therefore needs a NEW run (re-run Step 6), never an append.

Every planned call is written to `raw_responses.jsonl` BEFORE it runs (write-ahead, flushed + fsynced), then closed with its outcome: generated, exception, interrupted or aborted. If the run stops early or you interrupt it, every remaining planned call is recorded as skipped with the reason, the partial log is preserved, and the interruption is re-raised.

In [ ]:
import time, traceback
from PIL import Image

if not BUNDLE_VERIFIED:
    raise RuntimeError("bundle not verified")
# Re-verify immediately before inference: nothing may have changed since Step 4.
bundle = pilot_lib.verify_bundle(BUNDLE_DIR, EXPECTED_MANIFEST_SHA256, EXPECTED_PILOT_LIB_SHA256)
# Fail-closed gate (raises RunGateError and touches nothing if anything is wrong; otherwise claims the folder).
pilot_lib.assert_ready_for_inference(OUT_DIR, run_meta, RUN_ID)

MAX_NEW_TOKENS = pilot_lib.MAX_NEW_TOKENS
RAW_LOG = OUT_DIR / pilot_lib.RAW_LOG_NAME
log = pilot_lib.make_logger(OUT_DIR)

try:
    IMAGE_TOKEN_ID = processor.tokenizer.convert_tokens_to_ids("<image_soft_token>")
    if IMAGE_TOKEN_ID is None or IMAGE_TOKEN_ID == getattr(processor.tokenizer, "unk_token_id", None):
        IMAGE_TOKEN_ID = None
except Exception:
    IMAGE_TOKEN_ID = None
_eos = model.generation_config.eos_token_id
EOS_IDS = [] if _eos is None else (list(_eos) if isinstance(_eos, (list, tuple)) else [_eos])

def load_image(case):
    return Image.open(BUNDLE_DIR / "images" / case["filename"]).convert("RGB")

def generate_once(image, prompt_text):
    messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt_text}]}]
    inputs = processor.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True, return_dict=True, return_tensors="pt",
    ).to(model.device, dtype=model.dtype if hasattr(model, "dtype") else torch.bfloat16)
    input_len = int(inputs["input_ids"].shape[-1])
    n_image_tokens = int((inputs["input_ids"] == IMAGE_TOKEN_ID).sum()) if IMAGE_TOKEN_ID is not None else None
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, num_beams=1)
    ids = [int(t) for t in out[0][input_len:].tolist()]
    return {
        "raw_response": processor.decode(ids, skip_special_tokens=True),
        "raw_response_with_special_tokens": processor.decode(ids, skip_special_tokens=False),
        "generated_token_ids": ids,
        "n_generated_tokens": len(ids),
        "input_token_count": input_len,
        "image_token_count": n_image_tokens,
        "finish_reason": pilot_lib.determine_finish_reason(ids, EOS_IDS, MAX_NEW_TOKENS),
        "eos_token_ids": EOS_IDS,
    }

# Records every planned call write-ahead; KeyboardInterrupt is recorded, the partial log is kept, then re-raised.
outcome = pilot_lib.run_planned_calls(bundle["cases"], OUT_DIR, RUN_ID, load_image, generate_once, log=log)
print(outcome)

## Step 8 - Reconcile the log, parse, and summarise (also works after an interrupted or partial run)
Every planned call appears in `results.csv` (skipped and unfinished ones included, with their reason). Scored and exploratory cases are written to separate files and summarised separately; nothing is pooled and no accuracy aggregate is computed.

In [ ]:
RAW_LOG = OUT_DIR / pilot_lib.RAW_LOG_NAME  # always the CURRENT run folder, never a variable left over from an earlier run
events = pilot_lib.read_jsonl(RAW_LOG)
accounting = pilot_lib.verify_run_accounting(events, expected_run_id=RUN_ID)
rows = pilot_lib.build_result_rows(events, bundle["manifest"], expected_run_id=RUN_ID)
pilot_lib.write_split_results(OUT_DIR, rows)

non_ok = [r for r in rows if r["parse_status"] != "ok"]
with open(OUT_DIR / "parse_failures.jsonl", "w", encoding="utf-8") as f:
    for r in non_ok:
        f.write(json.dumps(r, ensure_ascii=False, sort_keys=True) + "\n")

summary = pilot_lib.descriptive_summary(rows, accounting)
pilot_lib.atomic_write_json(OUT_DIR / "summary.json", summary)
meta_now = pilot_lib.update_run_metadata(OUT_DIR, {"n_non_ok_rows": len(non_ok), "log_fully_accounted": accounting["all_accounted"]})

print(pilot_lib.BANNER)
print("run state:", meta_now["status"], "| fully accounted:", accounting["all_accounted"], "| planned:", accounting["n_planned"])
print({k: v for k, v in accounting.items() if k.startswith("n_")})
print(json.dumps(summary["groups"], indent=2))
for r in rows:
    print(r["seq"], r["group"], r["case_id"], r["experiment"], r["call_status"], r["parse_status"], r["major_mm_pred"], r["minor_mm_pred"], r["skip_reason"] or "")

## Step 9 - Download the run folder
Place the unzipped folder at `validation/phase0_pilot5/runs/<run_id>/` in the repo. Do not put it in `outputs/` or `results/`.

In [ ]:
import shutil
archive = shutil.make_archive(str(OUT_DIR), "zip", OUT_DIR)
print("created", archive)
files.download(archive)